# 🚁 Thesis: Trening Detektora Dłoni YOLO (Etap 1)

Niniejszy notatnik służy do wytrenowania **modelu detekcji dłoni w kadrze z drona** w chmurze **Google Colab** (z darmowym lub płatnym GPU NVIDIA T4/A100).

### 🧠 Kontekst Architektury (Dwustopniowy Detektor):
1. **Etap 1 (Ten Model):** YOLO Nano (np. `yolo11n` / `yolov8n`) — lokalizuje bounding box dłoni (`hand`).
2. **Etap 2:** Customowa sieć CNN / ResNet — klasyfikuje wycięty fragment dłoni (`crop`) do konkretnego gestu (`ok`, `shaka`, `v`, `openhand` itp.).

## 1. ⚡ Sprawdzenie dostępnego GPU
Upewnij się, że w menu Colaba masz włączoną akcelerację: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
!nvidia-smi

## 2. 📦 Instalacja biblioteki Ultralytics (YOLO)

In [ ]:
!pip install -q ultralytics

## 3. 📂 Montowanie Google Drive i rozpakowanie zbioru danych

> **Wskazówka wydajnościowa:**  
> Nie trenuj bezpośrednio na plikach leżących na zamontowanym Google Drive (odczyt tysięcy małych plików przez sieć spowalnia trening 5-krotnie!).  
> Najszybsza metoda: skopiuj pojedynczy plik `yolo_dataset.zip` z Drive na lokalny szybki dysk SSD kontenera (`/content/`) i rozpakuj go tam.

In [ ]:
from google.colab import drive
import os
import zipfile

# 1. Montowanie Google Drive
drive.mount('/content/drive')

# 2. Ścieżka do spakowanego datasetu na Google Drive (dostosuj jeśli zapisałeś w podfolderze)
zip_source = '/content/drive/MyDrive/yolo_dataset.zip'
dataset_dest = '/content/dataset'

if os.path.exists(zip_source):
    print(f'Rozpakowywanie {zip_source} na dysk lokalny Colaba ({dataset_dest})...')
    os.makedirs(dataset_dest, exist_ok=True)
    with zipfile.ZipFile(zip_source, 'r') as z:
        z.extractall(dataset_dest)
    print('✅ Zbiór danych pomyślnie rozpakowany na lokalnym dysku SSD!')
else:
    print(f'⚠️ Nie znaleziono pliku {zip_source}.')
    print('Możesz wgrać plik yolo_dataset.zip ręcznie przez panel plików po lewej stronie Colaba.')

## 4. 🚀 Trening modelu YOLO Nano (Etap 1: Detekcja dłoni)

In [ ]:
from ultralytics import YOLO

# Model bazowy: YOLO11 Nano (najlżejszy, idealny na drona / edge computing)
# Możesz też wybrać 'yolov8n.pt'
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,           # GPU NVIDIA w Colabie
    patience=25,        # Wczesne zatrzymanie jeśli brak poprawy przez 25 epok
    lr0=0.01,
    workers=4,
    project='/content/runs',
    name='hand_detector',
    save=True,
    plots=True,
    verbose=True
)

## 5. 📊 Podgląd wyników i wykresów zbieżności

In [ ]:
from IPython.display import Image, display
import glob

run_dir = '/content/runs/hand_detector'

# Wyświetl wykresy uczenia (straty, precyzja, recall, mAP)
results_png = f'{run_dir}/results.png'
if os.path.exists(results_png):
    print('Wykresy uczenia (Losses & Metrics):')
    display(Image(results_png))

# Wyświetl macierz pomyłek
confusion_png = f'{run_dir}/confusion_matrix.png'
if os.path.exists(confusion_png):
    print('Macierz pomyłek (Confusion Matrix):')
    display(Image(confusion_png))

# Wyświetl przykładowe predykcje na zbiorze walidacyjnym
val_preds = glob.glob(f'{run_dir}/val_batch*_pred.jpg')
if val_preds:
    print('Przykładowe predykcje walidacyjne:')
    display(Image(val_preds[0]))

## 6. 💾 Zapisanie najlepszego modelu (best.pt) na Google Drive

In [ ]:
import shutil

best_weights = '/content/runs/hand_detector/weights/best.pt'
drive_target_dir = '/content/drive/MyDrive/thesis_weights'
os.makedirs(drive_target_dir, exist_ok=True)

if os.path.exists(best_weights):
    target_file = f'{drive_target_dir}/yolo_hand_detector_best.pt'
    shutil.copy2(best_weights, target_file)
    print(f'🎉 Sukces! Zapisano najlepsze wagi do: {target_file}')
    print('Możesz teraz pobrać ten plik na swój komputer i używać w etapie 1 pipelineu.')
else:
    print('⚠️ Nie znaleziono pliku best.pt!')